In [0]:
--Find top 5 stocks by average return for the last 40 days

with base as (
  select  
    ticker,
    trade_date,
    close,
    open,
    high,
    low,
    volume,
    lag(close) over (
      partition by ticker
      order by trade_date
    ) as prev_close
  from analytics.stock_prices
),

returns as (
  select *,
  (close - prev_close) / prev_close as daily_return
  from base
  where prev_close is not null
),

numbered as (
  select *,
    row_number() over (
      partition by ticker
      order by trade_date
    ) as rn
  from returns
),

max_rank as (
  select *,
    max(rn) over (
      partition by ticker
    ) as max_rn
  from numbered
),

filtered as (
  select *
  from max_rank
  where rn > max_rn - 40
),

final as (
  select
    ticker,
    max(case when rn = max_rn then close end) as last_close,
    max(case when rn = max_rn - 39 then close end) as first_close
  from filtered
  group by ticker
)

select *,
  (last_close / first_close) - 1 as cum_return
from final
order by cum_return desc
limit 5;

--Find stocks with increasing volume 3 days in a row (dynamic version)

with base as (
  select *,
    lag(volume) over (
      partition by ticker
      order by trade_date
    ) as prev_volume
  from analytics.stock_prices
),

daily_changes as (
  select *,
    (volume - prev_volume) / prev_volume as pct_change
  from base
  where prev_volume is not null
),

direction_flag as (
  select *,
  case
    when pct_change > 0 then "UP"
    when pct_change < 0 then "DOWN"
    else "Flat"
  end as direction
  from daily_changes
),

numbered as (
  select *,
    row_number() over (
      partition by ticker
      order by trade_date
    ) as rn1,
    row_number() over (
      partition by ticker, direction
      order by trade_date
    ) as rn2
  from direction_flag
),

grouped as (
  select *,
    rn1 - rn2 as streak_group
  from numbered
),

streak_counts as (
  select 
    ticker,
    streak_group,
    count(*) as streak_length,
    min(trade_date) as start_date,
    max(trade_date) as end_date
  from grouped
  where direction = 'UP'
  group by ticker, streak_group
)

select *
from streak_counts
where streak_length >= 3
order by ticker, start_date;


--Find stocks with increasing volume 3 days in a row (strict)

with base as (
  select *,
    lag(volume, 1) over (
      partition by ticker
      order by trade_date
    ) as prev_volume_1,
    lag(volume, 2) over (
      partition by ticker
      order by trade_date
    ) as prev_volume_2
  from analytics.stock_prices
)

select *
from base
where volume > prev_volume_1
and prev_volume_1 > prev_volume_2;

--Find longest winning streak per stock

with base as (
  select *,
    lag(close) over (
      partition by ticker
      order by trade_date
    ) as prev_close
  from analytics.stock_prices
),

flagged as (
  select *,
    case
     when close > prev_close then 1
     else 0
    end as is_win
  from base
  where prev_close is not null
),

numbered as (
  select *,
    row_number() over (
      partition by ticker
      order by trade_date
    ) as rn1,
    row_number() over (
      partition by ticker, is_win
      order by trade_date
    ) as rn2
  from flagged
),

grouped as (
  select *,
  rn1 - rn2 as streak_group
  from numbered
),

streaks as (
  select
    ticker,
    streak_group,
    count(*) as streak_length
  from grouped
  where is_win = 1
  group by ticker, streak_group
)

select ticker, max(streak_length) as max_streak
from streaks
group by ticker
order by max_streak desc;

--Monthly Return summary pivot-style output

with base as (
  select *,
    lag(close) over (
      partition by ticker
      order by trade_date
    ) as prev_close
  from analytics.stock_prices
),

returns as (
  select *,
    (close - prev_close) / prev_close as daily_return
  from base
  where prev_close is not null
),

monthly_groups as (
  select 
    ticker,
    date_format(trade_date, 'yyyy-MM') as yr_month,
    daily_return
  from returns
),

pivot as (
  select *
  from monthly_groups
  pivot(
    exp(sum(ln(1 + daily_return))) - 1
    for yr_month in ('2025-12' as Dec25, '2026-01' as Jan26, '2026-02' as Feb26, '2026-03' as March26 ))
)

select *
from pivot;

--Monthly Return Summary Pivot-style report (grouping instead of using pivot function)

with base as (
  select *,
    lag(close) over (
      partition by ticker
      order by trade_date
    ) as prev_close
  from analytics.stock_prices
),

returns as (
  select *,
    (close - prev_close) / prev_close as daily_return
  from base
  where prev_close is not null
),

monthly_groups as (
  select
    ticker,
    date_format(trade_date, 'yyyy-MM') as year_month,
    daily_return
  from returns
)

select 
  ticker,
  year_month,
  exp(sum(ln(1 + daily_return))) - 1 as monthly_return
from monthly_groups
group by ticker, year_month
order by ticker, year_month;






Databricks visualization. Run in Databricks to view.